# Explanation shift, part 1: how far each explanation moves

**This analysis is exploratory.** It was specified on 9 October 2026, after the registered results of all
five studies were known, and no registration names it. Nothing it computes tests a registered hypothesis.

In Studies 2, 4 and 5 each participant explained the same source artwork three times for each rating
question: once for the second work that left it unchanged (None), once for a slight change (Slight) and
once for a dramatic change (Dramatic). If an AI source taints the second work, participants told that the
source was made by prompting an AI system should keep writing about where the source came from however much
Alex changed it, so their explanations should move less from None to Dramatic than those of participants
told that Alex made the source with design and sketching tools. If participants judge what Alex added, the explanations of the
two arms should move about as far.

This notebook measures that movement. For every participant, source image and question with an
explanation at all three levels, it compares the explanations at each pair of levels (None and Slight,
Slight and Dramatic, None and Dramatic) in two ways.

- **Embedding distance.** A sentence-embedding model is a network trained beforehand on other text that
  turns a sentence into a list of numbers, so that sentences with similar meanings get similar lists. The
  locally stored model all-mpnet-base-v2 turns each explanation into 768 numbers. The embedding distance is
  1 minus the cosine similarity of the two lists, where the cosine similarity is 1 when the lists point
  the same way and 0 when they are unrelated.
- **Word distance.** A content word is a word left once the fixed list of function words, such as *the*
  and *of*, is removed. The Jaccard overlap of two explanations is the number of content words both use
  divided by the number either uses. The word distance is 1 minus that overlap.

Each distance has a stranger baseline: the mean distance between this participant's explanation at the
first level and the explanations other participants in the same study wrote about the same image and
question at the second level. Two people writing about the same picture share words because the picture
supplies them, and the baseline is the distance to expect when only the picture is shared. It is the
baseline of the registered word-overlap check in `shared/R/text_checks.R`.

**The measurement is blind by construction.** From each study's long data file it reads six columns: the
participant, the source image, the modification level, the rating question, the item name and the text.
The item name, such as `1FigureMaxCreat`, is read only to check the image, the level and the question of
each row. The notebook never reads the Human or AI label, the time condition or the rating, and the file it
writes carries none of them.
`ExplanationShift_02_Analysis.Rmd` joins the label on.

No person and no language model labels, codes or judges any response. The embedding model turns each
response into numbers and does nothing else.

**Where it runs.** Locally, on the Jupyter kernel named `python312` (the entry that tells Jupyter which
Python to start), with the model read from the copy already on the machine and nothing downloaded. From
this folder:

    HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 jupyter nbconvert --to notebook --execute --inplace \
      --ExecutePreprocessor.kernel_name=python312 ExplanationShift_01_Measure.ipynb

It also runs in Google Colab, Google's service that runs a notebook on its own machines from a web
browser. Open this notebook from Google Drive with Colab and run every cell. The first code cell attaches
Google Drive, and the embedding cell may download the model once.

---

# 0. Setup

The first code cell finds this folder. In Colab it mounts Google Drive, which attaches the Drive to the
machine so that its files are visible under `/content/drive`, and Colab asks for permission once. On a
local machine it skips the mount and uses the folder the notebook runs from. The environment variable
`EXPLANATION_SHIFT_DIR`, a setting the shell passes to the program, can name the folder instead, and
`STUDY_SHARED_PYTHON` can name the folder of the shared module.

Every path the notebook prints is relative to the pipeline folder, so the output names no account and no
folder particular to one machine.

A seed is the number that starts a random-number generator, so that a rerun draws the same numbers. No
step of this notebook draws a random number. The cell still sets 20260903, the seed of the text code of
Studies 3 to 5, through the shared module's `set_text_seed()`, so that any random step it reaches starts
from that seed, and the environment table records it.

In [1]:
import datetime
import hashlib
import os
import platform
import sys
import time
import warnings

# Google Colab has a module of its own, so importing it tells the two places apart.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Locally the Hugging Face libraries are kept off the internet, so the model
# is read from the stored copy and nothing is downloaded.
if not IN_COLAB:
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def short_warning(message, category, filename, lineno, line=None):
    # A warning names the file it came from without the file's folder.
    return "%s: %s (%s, line %d)\n" % (category.__name__, message, os.path.basename(filename), lineno)


warnings.formatwarning = short_warning

import numpy as np
import pandas as pd
from scipy import sparse

# In Colab, Google Drive is mounted before any file is read.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# This folder's place below the top of Google Drive.
DRIVE_TAIL = os.path.join("DGK Lab", "Reasoning Lab", "DGK Lab - Copyrightability and AI Art",
                          "Analysis", "R Analysis Pipeline", "Cross-study", "explanation_shift")

# The candidate folders, in the order they are tried: the one
# EXPLANATION_SHIFT_DIR names, the two places Colab can show Google Drive, and
# the folder the notebook runs from.
CANDIDATES = [os.environ.get("EXPLANATION_SHIFT_DIR", "")]
if IN_COLAB:
    CANDIDATES += [os.path.join("/content/drive/MyDrive", DRIVE_TAIL),
                   os.path.join("/content/drive/My Drive", DRIVE_TAIL)]
CANDIDATES.append(os.getcwd())


def is_this_folder(folder):
    # This folder sits two levels below the pipeline folder, which holds the shared module.
    return bool(folder) and os.path.exists(
        os.path.join(folder, os.pardir, os.pardir, "shared", "python", "study_text.py"))


HERE = next((os.path.abspath(f) for f in CANDIDATES if is_this_folder(f)), None)
if HERE is None:
    raise FileNotFoundError(
        "Could not find the explanation_shift folder. In Colab, check that Google Drive mounted and "
        "that the folder is at My Drive/" + DRIVE_TAIL + ". Locally, run the notebook from its own "
        "folder or set EXPLANATION_SHIFT_DIR to that folder.")

PIPELINE_DIR = os.path.normpath(os.path.join(HERE, os.pardir, os.pardir))
SHARED_PYTHON = os.path.abspath(os.environ.get("STUDY_SHARED_PYTHON",
                                               os.path.join(PIPELINE_DIR, "shared", "python")))
sys.dont_write_bytecode = True
sys.path.insert(0, SHARED_PYTHON)
import study_text

show = study_text.display_path

# The seed of the text code of Studies 3 to 5. No step below draws a random number.
TEXT_SEED = 20260903
TEXT_SEEDS = study_text.set_text_seed(TEXT_SEED)

NLP_DIR = os.path.join(HERE, "nlp")
TABLES_DIR = os.path.join(HERE, "tables")
os.makedirs(NLP_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
PAIRS_PATH = os.path.join(NLP_DIR, "explanation_shift_pairs.csv")
ENVIRONMENT_PATH = os.path.join(TABLES_DIR, "explanation_shift_measure_environment.csv")
MANIFEST_PATH = os.path.join(TABLES_DIR, "explanation_shift_measure_run_manifest.csv")

# The previous run's manifest is deleted before this run writes anything, so
# a manifest on disk means the run that wrote it reached the last cell.
if os.path.exists(MANIFEST_PATH):
    os.remove(MANIFEST_PATH)

RUN_ID = study_text.new_run_id()

# The status every table of this analysis carries in its first column.
EXPLORATORY = ("exploratory: specified on 9 October 2026, after the registered results of all five "
               "studies were known; no registration names it")


def file_sha256(path):
    # The SHA-256 checksum of a file's bytes: a 64-character string that
    # changes if any byte of the file changes.
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


print("Running in Colab: ", IN_COLAB)
print("This folder:      ", show(HERE))
print("Shared module:    ", show(study_text.__file__))
print("Writing data to:  ", show(NLP_DIR))
print("Writing tables to:", show(TABLES_DIR))
print("Run identifier:   ", RUN_ID)
print("Seeds:            ", TEXT_SEEDS)

Running in Colab:  False
This folder:       Cross-study/explanation_shift
Shared module:     shared/python/study_text.py
Writing data to:   Cross-study/explanation_shift/nlp
Writing tables to: Cross-study/explanation_shift/tables
Run identifier:    20261014T191240Z-2e6d80f3
Seeds:             {'stability_refits_first': 20260903, 'stability_refits_last': 20260912, 'final_topic_fit': 20260903, 'kmeans_grouping': 20260903}


---

# 1. The blind slice

The blind slice is the only table the measurement sees. From each study's long data file, `pandas` (the
Python library that holds tables) loads six columns, so the label, the time condition and the rating
never enter this notebook's memory. The cell first reads the file's column names alone and checks that the
columns it leaves out are there to be left out.

The files name their columns differently. Study 4 calls the source image `source` and the Human or AI
label `label`. Studies 2 and 5 call the source image `image` and the label `source`. Each study's image
column is read under its own name, and every value read as an image must be an image name such as
`3NonFigure`, so a label cannot pass as an image. The modification level is read from `modification`.

The item name spells out the image, the level and the question, in that order: `1FigureMaxCreat` is the
creativity question about `1Figure` at the dramatic change. The item names call the slight change `Min` and
the dramatic change `Max`. On every row the item name must agree with the image, the level and the question,
with `Min` read as Slight and `Max` as Dramatic, or the notebook stops. The item name is then dropped, so
each pair the notebook builds is known to join one participant's explanations of one image and one question.

Each study leaves its two six-point transformativeness items out of its main models, because their answer
scale has six points instead of five. They are the transformativeness questions about source image
`1NonFigure` at None and at Slight. `Study2_01_Quant.Rmd` drops them before it writes the long data file,
and Studies 4 and 5 keep their 140 and 274 rows in the file and leave them out of their models. They are
dropped here in all three studies, which leaves no complete set of three levels for transformativeness of
`1NonFigure`.

In [2]:
# Each study's long data file, the name of its source-image column, the
# columns that carry its label, time condition and rating (left unread), and
# the counts its own files state.
STUDIES = {
    "Study 2": dict(file=os.path.join("Study 2", "study2_long_with_text.csv"), image="image",
                    not_read=["source", "rating", "rating_label"],
                    rows=20659, people=235, six_point_rows=0),
    "Study 4": dict(file=os.path.join("Study 4", "study4_long_with_text.csv"), image="source",
                    not_read=["label", "source_time", "rating", "rating_label"],
                    rows=6318, people=117, six_point_rows=140),
    "Study 5": dict(file=os.path.join("Study 5", "study5_long_with_text.csv"), image="image",
                    not_read=["source", "editing_time", "rating", "rating_01", "rating_label"],
                    rows=12042, people=223, six_point_rows=274),
}
LEVEL_NAMES = {"None": "None", "Min": "Slight", "Slight": "Slight", "Max": "Dramatic", "Dramatic": "Dramatic"}
# How the item names spell the level and the question.
ITEM_LEVELS = {"None": "None", "Min": "Slight", "Max": "Dramatic"}
ITEM_QUESTIONS = {"Trans": "transformativeness", "Ess": "essence_change", "Creat": "creativity"}
LEVELS = ["None", "Slight", "Dramatic"]
DIMENSIONS = ["transformativeness", "essence_change", "creativity"]
SIX_POINT_IMAGE, SIX_POINT_DIMENSION, SIX_POINT_LEVELS = "1NonFigure", "transformativeness", ["None", "Slight"]

frames = []
INPUTS = {}
for study, spec in STUDIES.items():
    path = os.path.join(PIPELINE_DIR, spec["file"])
    read_columns = ["id", spec["image"], "modification", "dimension", "item", "response_text"]
    header = list(pd.read_csv(path, nrows=0).columns)
    assert set(spec["not_read"]) <= set(header), "%s lacks a column the check expects." % study
    assert not set(spec["not_read"]) & set(read_columns), "A label, time or rating column would be read."

    raw = pd.read_csv(path, usecols=read_columns, dtype=str, keep_default_na=False)
    assert sorted(raw.columns) == sorted(read_columns), "Unexpected columns were read."
    INPUTS[path] = file_sha256(path)

    frame = pd.DataFrame({
        "study": study,
        "id": raw["id"],
        "image": raw[spec["image"]],
        "level": raw["modification"].map(LEVEL_NAMES),
        "dimension": raw["dimension"],
        "response_text": raw["response_text"],
    })
    assert frame["level"].notna().all(), "%s has a modification level outside None, Slight and Dramatic." % study
    assert frame["image"].str.fullmatch(r"\d(Figure|NonFigure)").all(), "%s has a value that is not an image." % study
    assert set(frame["dimension"]) == set(DIMENSIONS), "%s has an unexpected question." % study
    assert len(frame) == spec["rows"] and frame["id"].nunique() == spec["people"], \
        "%s does not hold the %d rows from %d participants its own files state." % (study, spec["rows"], spec["people"])

    # The item name, such as 1FigureMaxCreat, repeats the image, the level and
    # the question. It must agree with all three on every row.
    parts = raw["item"].str.extract(r"^(\d(?:Figure|NonFigure))(None|Min|Max)(Trans|Ess|Creat)$")
    assert parts.notna().all().all(), "%s has an item name of an unexpected form." % study
    assert (parts[0] == frame["image"]).all(), "%s: an item name disagrees with its image." % study
    assert (parts[1].map(ITEM_LEVELS) == frame["level"]).all(), "%s: an item name disagrees with its level." % study
    assert (parts[2].map(ITEM_QUESTIONS) == frame["dimension"]).all(), \
        "%s: an item name disagrees with its question." % study

    six_point = ((frame["image"] == SIX_POINT_IMAGE) & (frame["dimension"] == SIX_POINT_DIMENSION)
                 & frame["level"].isin(SIX_POINT_LEVELS))
    assert int(six_point.sum()) == spec["six_point_rows"], "%s: unexpected number of six-point rows." % study
    frame = frame[~six_point]
    assert (frame["response_text"].str.strip() != "").all(), "%s has an empty explanation." % study
    frames.append(frame)
    print("%s: %s | %6d rows read, %3d six-point rows dropped, %6d kept | %d participants"
          % (study, show(path), spec["rows"], int(six_point.sum()), len(frame), frame["id"].nunique()))

print("Every item name agrees with its row's image, level and question.")
blind = pd.concat(frames, ignore_index=True)
del frames, raw, frame, six_point, parts

# The slice holds the six columns below and nothing else. A column named for
# a label, a time condition or a rating would stop the notebook here.
FORBIDDEN = {"source", "label", "condition", "arm", "type", "rating", "rating_01", "rating_label",
             "source_time", "editing_time", "label_mode", "k_points"}
assert list(blind.columns) == ["study", "id", "image", "level", "dimension", "response_text"]
assert not FORBIDDEN & {c.lower() for c in blind.columns}
RESPONSE_KEY = ["study", "id", "image", "level", "dimension"]
assert not blind.duplicated(RESPONSE_KEY).any(), "Some participant explained one second work twice on one question."
print()
print("Explanations in the slice:", len(blind), "| columns:", list(blind.columns))

Study 2: Study 2/study2_long_with_text.csv |  20659 rows read,   0 six-point rows dropped,  20659 kept | 235 participants
Study 4: Study 4/study4_long_with_text.csv |   6318 rows read, 140 six-point rows dropped,   6178 kept | 117 participants
Study 5: Study 5/study5_long_with_text.csv |  12042 rows read, 274 six-point rows dropped,  11768 kept | 223 participants
Every item name agrees with its row's image, level and question.

Explanations in the slice: 38605 | columns: ['study', 'id', 'image', 'level', 'dimension', 'response_text']


---

# 2. The complete sets of three

A set of three is one participant's three explanations of one source image on one question, at None,
Slight and Dramatic. Only complete sets are measured. The counts follow from each study's design.

- Study 2: 235 participants each saw all 10 source images, which makes 7,050 participant, image and
  question cells. The 235 cells of `1NonFigure` transformativeness lack None and Slight, and the seven
  participants who lack the trial at Slight for `1Figure` lack it on all three questions, which leaves 6,794
  complete sets.
- Study 4: 117 participants each saw 6 source images, which makes 2,106 cells, less the 70 for the
  participants who saw `1NonFigure`: 2,036.
- Study 5: 223 participants each saw 6 source images, which makes 4,014 cells, less the 137 for the
  participants who saw `1NonFigure`: 3,877.

The stranger baselines draw on every explanation in the slice, whether or not it belongs to a complete set,
as the registered overlap check draws on every other participant who wrote about the same stimulus.

In [3]:
SET_KEY = ["study", "id", "image", "dimension"]
levels_per_set = blind.groupby(SET_KEY)["level"].nunique()
complete_sets = levels_per_set[levels_per_set == 3].index
in_complete = blind.set_index(SET_KEY).index.isin(complete_sets)

counts = (blind[in_complete].drop_duplicates(SET_KEY)
          .groupby(["study", "dimension"])
          .agg(complete_sets=("id", "size"), participants=("id", "nunique"))
          .reindex(pd.MultiIndex.from_product([list(STUDIES), DIMENSIONS], names=["study", "dimension"])))
print(counts.to_string())
print()
sets_per_study = blind[in_complete].drop_duplicates(SET_KEY).groupby("study").size().to_dict()
print("Complete sets by study:", sets_per_study, "| all:", len(complete_sets))
assert sets_per_study == {"Study 2": 6794, "Study 4": 2036, "Study 5": 3877}

                            complete_sets  participants
study   dimension                                      
Study 2 transformativeness           2108           235
        essence_change               2343           235
        creativity                   2343           235
Study 4 transformativeness            632           117
        essence_change                702           117
        creativity                    702           117
Study 5 transformativeness           1201           223
        essence_change               1338           223
        creativity                   1338           223

Complete sets by study: {'Study 2': 6794, 'Study 4': 2036, 'Study 5': 3877} | all: 12707


---

# 3. Content words and the word distance

Each explanation's content words come from the shared module's `tokenize()`, which keeps every run of two
or more letters (apostrophes included) in lower case and removes the shared list `FUNCTION_WORDS`. The
words of the task, such as *creativity* and *artwork*, stay in, as they do in the registered word-overlap
check. An explanation's word set holds each content word once.

The word distance of a pair is 1 minus the Jaccard overlap of the two word sets. An explanation with no
content word, such as *same*, has nothing to overlap, so a pair that includes one has no word distance, and
such an explanation is never anyone's stranger. The registered overlap check treats it the same way.

The distances are computed for each study, source image and question at once, as a table of every
first-level explanation against every second-level explanation. The participant's own pair is one cell of
that table, and the stranger baseline is the mean of the other cells in the participant's row. The cell
ends by recomputing 500 own distances, spread evenly through the table, one at a time from the word sets,
and they must agree.

In [4]:
assert study_text.IGNORE == set(study_text.FUNCTION_WORDS), "tokenize() no longer removes FUNCTION_WORDS."
blind["words"] = [frozenset(study_text.tokenize(text)) for text in blind["response_text"]]
blind["n_words"] = blind["words"].map(len)
print("Explanations with no content word:", int((blind["n_words"] == 0).sum()), "of", len(blind))

PAIRS = [("None", "Slight"), ("Slight", "Dramatic"), ("None", "Dramatic")]
PAIR_NAMES = {pair: "%s-%s" % pair for pair in PAIRS}
complete_set_ids = {key: set(group["id"]) for key, group in
                    blind[in_complete].groupby(["study", "image", "dimension"])}


def word_distance_matrix(sets_a, sets_b):
    # Rows are the first-level explanations and columns the second-level ones.
    # A cell holds 1 minus the Jaccard overlap, or NaN when either side has no
    # content word.
    vocabulary = {w: k for k, w in enumerate(sorted(set().union(*sets_a, *sets_b)))}

    def binary(sets):
        rows = [r for r, words in enumerate(sets) for _ in words]
        cols = [vocabulary[w] for words in sets for w in words]
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(vocabulary)))

    shared = (binary(sets_a) @ binary(sets_b).T).toarray()
    size_a = np.array([len(s) for s in sets_a], dtype=float)
    size_b = np.array([len(s) for s in sets_b], dtype=float)
    either = size_a[:, None] + size_b[None, :] - shared
    distance = 1.0 - shared / np.where(either > 0, either, 1.0)
    distance[(size_a == 0)[:, None] | (size_b == 0)[None, :]] = np.nan
    return distance


def own_and_stranger(distance, ids_a, ids_b, keep):
    # For each first-level explanation whose participant has a complete set:
    # the distance to the same participant's second-level explanation, and the
    # mean distance to every other participant's second-level explanation that
    # has a distance.
    other = ids_a[:, None] != ids_b[None, :]
    usable = other & ~np.isnan(distance)
    n_strangers = usable.sum(axis=1)
    total = np.where(usable, distance, 0.0).sum(axis=1)
    stranger = np.where(n_strangers > 0, total / np.maximum(n_strangers, 1), np.nan)
    column_of = {pid: j for j, pid in enumerate(ids_b)}
    rows = np.array([i for i, pid in enumerate(ids_a) if pid in keep], dtype=int)
    own = np.array([distance[i, column_of[ids_a[i]]] for i in rows])
    return rows, own, stranger[rows], n_strangers[rows]


word_rows = []
for (study, image, dimension), group in blind.groupby(["study", "image", "dimension"], sort=True):
    keep = complete_set_ids.get((study, image, dimension), set())
    if not keep:
        continue
    for first, second in PAIRS:
        a = group[group["level"] == first]
        b = group[group["level"] == second]
        distance = word_distance_matrix(list(a["words"]), list(b["words"]))
        rows, own, stranger, n_strangers = own_and_stranger(
            distance, a["id"].to_numpy(), b["id"].to_numpy(), keep)
        size_of_b = dict(zip(b["id"], b["n_words"]))
        for k, i in enumerate(rows):
            pid = a["id"].iloc[i]
            word_rows.append(dict(study=study, id=pid, image=image, dimension=dimension,
                                  pair=PAIR_NAMES[(first, second)],
                                  word_distance=own[k], word_stranger=stranger[k],
                                  word_strangers_used=int(n_strangers[k]),
                                  words_a=int(a["n_words"].iloc[i]), words_b=int(size_of_b[pid])))
word_pairs = pd.DataFrame(word_rows)
del word_rows
assert len(word_pairs) == 3 * len(complete_sets)

# The same distances, one at a time from the word sets.
words_of = blind.set_index(RESPONSE_KEY)["words"]
first_of = {name: pair[0] for pair, name in PAIR_NAMES.items()}
second_of = {name: pair[1] for pair, name in PAIR_NAMES.items()}
for row in word_pairs.iloc[np.linspace(0, len(word_pairs) - 1, 500).astype(int)].itertuples(index=False):
    a = words_of[(row.study, row.id, row.image, first_of[row.pair], row.dimension)]
    b = words_of[(row.study, row.id, row.image, second_of[row.pair], row.dimension)]
    expected = np.nan if not a or not b else 1.0 - len(a & b) / len(a | b)
    assert (np.isnan(expected) and np.isnan(row.word_distance)) or abs(expected - row.word_distance) < 1e-12
print("Pairs:", len(word_pairs), "| with no word distance:", int(word_pairs["word_distance"].isna().sum()),
      "| strangers per first-level explanation: median",
      int(word_pairs.loc[word_pairs["words_a"] > 0, "word_strangers_used"].median()))
print("500 own word distances recomputed one at a time: all agree")

Explanations with no content word: 588 of 38605


Pairs: 38121 | with no word distance: 1066 | strangers per first-level explanation: median 229
500 own word distances recomputed one at a time: all agree


---

# 4. Embeddings and the embedding distance

The model is `sentence-transformers/all-mpnet-base-v2`, loaded through the shared module's
`load_embedder()`, which reads it only from the local Hugging Face cache (the folder where these libraries
keep downloaded models) and runs it on the processor. In Colab the cell first downloads the model into
Colab's cache, from Hugging Face, the site that distributes it, and then loads it the same way. If the Colab
machine has a graphics card the model runs on it there, which changes the distances only in about the sixth
decimal place; the environment table records the device.

Each distinct text is embedded once, in a fixed order, and every explanation with that text gets the same
768 numbers, scaled to length 1 so that the cosine similarity of two lists is their dot product, the sum of
the products of their matching numbers. The cell
prints its progress. On a twelve-core Mac the 38,605 explanations, 29,296 distinct texts, took about two and
a half minutes, and the whole notebook about three.

The embedding distance of a pair is 1 minus the cosine similarity of its two lists. Two identical texts can
come out a few parts in ten million below zero from rounding, and such values are set to zero. Every
explanation has an embedding distance, including one with no content word. The stranger baseline is built
as in section 3, from every other participant's explanation of the same image and question at the second
level.

In [5]:
if IN_COLAB:
    # Colab may lack the library, and its cache starts empty.
    try:
        import sentence_transformers  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from huggingface_hub import snapshot_download
    snapshot_download(study_text.EMBEDDING_MODEL_NAME)

embedder, EMBEDDING_STATUS = study_text.load_embedder()
assert embedder is not None, "The embedding model did not load: %s" % EMBEDDING_STATUS
DEVICE = "cpu"
if IN_COLAB:
    import torch
    if torch.cuda.is_available():
        embedder = embedder.to("cuda")
        DEVICE = "cuda"
print("Embedding model:", study_text.EMBEDDING_MODEL_NAME, "|", EMBEDDING_STATUS, "| device:", DEVICE)

texts = sorted(set(blind["response_text"]))
row_of_text = {text: k for k, text in enumerate(texts)}
size = embedder.get_sentence_embedding_dimension()
vectors = np.empty((len(texts), size), dtype=np.float32)
CHUNK = 2048
started = time.time()
for first in range(0, len(texts), CHUNK):
    chunk = texts[first:first + CHUNK]
    vectors[first:first + len(chunk)] = embedder.encode(chunk, batch_size=64, normalize_embeddings=True,
                                                        show_progress_bar=False, device=DEVICE)
    print("Embedded %6d of %d distinct texts | %4.0f seconds" % (first + len(chunk), len(texts),
                                                                  time.time() - started), flush=True)
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4), "Some list is not of length 1."
vectors = vectors.astype(np.float64)
blind["vector_row"] = blind["response_text"].map(row_of_text)
print("Explanations:", len(blind), "| distinct texts:", len(texts), "| numbers per text:", size)

Embedding model: sentence-transformers/all-mpnet-base-v2 | loaded from the local copy | device: cpu


Embedded   2048 of 29296 distinct texts |   42 seconds


Embedded   4096 of 29296 distinct texts |   88 seconds


Embedded   6144 of 29296 distinct texts |  134 seconds


Embedded   8192 of 29296 distinct texts |  170 seconds


...

Embedded  24576 of 29296 distinct texts |  341 seconds


Embedded  26624 of 29296 distinct texts |  349 seconds


Embedded  28672 of 29296 distinct texts |  370 seconds


Embedded  29296 of 29296 distinct texts |  376 seconds


Explanations: 38605 | distinct texts: 29296 | numbers per text: 768


In [6]:
emb_rows = []
for (study, image, dimension), group in blind.groupby(["study", "image", "dimension"], sort=True):
    keep = complete_set_ids.get((study, image, dimension), set())
    if not keep:
        continue
    for first, second in PAIRS:
        a = group[group["level"] == first]
        b = group[group["level"] == second]
        similarity = vectors[a["vector_row"].to_numpy()] @ vectors[b["vector_row"].to_numpy()].T
        distance = np.clip(1.0 - similarity, 0.0, 2.0)
        rows, own, stranger, n_strangers = own_and_stranger(
            distance, a["id"].to_numpy(), b["id"].to_numpy(), keep)
        for k, i in enumerate(rows):
            emb_rows.append(dict(study=study, id=a["id"].iloc[i], image=image, dimension=dimension,
                                 pair=PAIR_NAMES[(first, second)], emb_distance=own[k],
                                 emb_stranger=stranger[k], emb_strangers_used=int(n_strangers[k])))
emb_pairs = pd.DataFrame(emb_rows)
del emb_rows
assert len(emb_pairs) == 3 * len(complete_sets)

# The same distances, one at a time from the two lists.
row_of_response = blind.set_index(RESPONSE_KEY)["vector_row"]
for row in emb_pairs.iloc[np.linspace(0, len(emb_pairs) - 1, 500).astype(int)].itertuples(index=False):
    va = vectors[row_of_response[(row.study, row.id, row.image, first_of[row.pair], row.dimension)]]
    vb = vectors[row_of_response[(row.study, row.id, row.image, second_of[row.pair], row.dimension)]]
    assert abs(min(max(1.0 - float(va @ vb), 0.0), 2.0) - row.emb_distance) < 1e-12
print("Pairs:", len(emb_pairs), "| strangers per first-level explanation: median",
      int(emb_pairs["emb_strangers_used"].median()), "| fewest", int(emb_pairs["emb_strangers_used"].min()))
print("500 own embedding distances recomputed one at a time: all agree")

Pairs: 38121 | strangers per first-level explanation: median 234 | fewest 67
500 own embedding distances recomputed one at a time: all agree


---

# 5. Export

The two halves are joined one row to one row on the study, participant, image, question and pair, and the
file `nlp/explanation_shift_pairs.csv` gets one row per complete set and pair of levels, with these columns:

| Column | What it holds |
|---|---|
| `analysis` | the status of the analysis: exploratory, specified on 9 October 2026 after the registered results were known, and named by no registration |
| `study` | Study 2, Study 4 or Study 5 |
| `id` | the participant |
| `image` | the source image |
| `dimension` | the rating question |
| `pair` | the two levels compared, first level first: `None-Slight`, `Slight-Dramatic` or `None-Dramatic` |
| `emb_distance` | 1 minus the cosine similarity of the participant's two explanations |
| `emb_stranger` | the mean embedding distance from the participant's first-level explanation to other participants' second-level explanations |
| `word_distance` | 1 minus the Jaccard overlap of the two content-word sets, empty when either has no content word |
| `word_stranger` | the mean word distance from the first-level explanation to other participants' second-level explanations, empty when the first-level explanation has no content word |
| `words_a`, `words_b` | the number of distinct content words in the first-level and the second-level explanation |

The file is read back from disk and checked before the run is recorded. The summary printed below is over
every participant and is split by nothing the label could touch. The environment table records the software,
the model and the device. The run manifest lists each input and output with its size and its SHA-256
checksum, a 64-character fingerprint of the file's exact bytes that changes if any byte changes, and it is
written last.

In [7]:
JOIN_KEY = ["study", "id", "image", "dimension", "pair"]
pairs = word_pairs.merge(emb_pairs, on=JOIN_KEY, how="inner", validate="one_to_one")
assert len(pairs) == len(word_pairs) == len(emb_pairs)

PAIR_COLUMNS = ["analysis", "study", "id", "image", "dimension", "pair", "emb_distance", "emb_stranger",
                "word_distance", "word_stranger", "words_a", "words_b"]
strangers_used = pairs[["emb_strangers_used", "word_strangers_used"]].describe().loc[["min", "50%", "max"]]
pairs["pair"] = pd.Categorical(pairs["pair"], categories=list(PAIR_NAMES.values()), ordered=True)
pairs["dimension"] = pd.Categorical(pairs["dimension"], categories=DIMENSIONS, ordered=True)
pairs = pairs.sort_values(["study", "id", "image", "dimension", "pair"], kind="mergesort")
# Like every table of this analysis, the file states its status in its first column.
pairs.insert(0, "analysis", EXPLORATORY)
pairs = pairs[PAIR_COLUMNS].astype({"pair": str, "dimension": str}).reset_index(drop=True)

# The contract of the file: its columns, no label, time or rating column,
# every embedding distance present and in range, and a word distance missing
# exactly where an explanation has no content word.
assert list(pairs.columns) == PAIR_COLUMNS
study_text.check_output(pairs, "pairs file")
assert not FORBIDDEN & {c.lower() for c in pairs.columns}
assert pairs[["emb_distance", "emb_stranger"]].notna().all().all()
assert pairs["emb_distance"].between(0, 2).all() and pairs["emb_stranger"].between(0, 2).all()
assert pairs["word_distance"].dropna().between(0, 1).all() and pairs["word_stranger"].dropna().between(0, 1).all()
assert (pairs["word_distance"].isna() == ((pairs["words_a"] == 0) | (pairs["words_b"] == 0))).all()
assert (pairs["word_stranger"].isna() == (pairs["words_a"] == 0)).all()
assert not pairs.duplicated(JOIN_KEY).any()

study_text.write_csv_atomic(PAIRS_PATH, pairs, RUN_ID)
on_disk = pd.read_csv(PAIRS_PATH, keep_default_na=True)
assert list(on_disk.columns) == PAIR_COLUMNS and len(on_disk) == len(pairs)
assert (on_disk["analysis"] == EXPLORATORY).all()
assert np.allclose(on_disk["emb_distance"], pairs["emb_distance"], rtol=0, atol=1e-15)
print("Written:", show(PAIRS_PATH), "|", len(pairs), "rows |", len(PAIR_COLUMNS), "columns")
print()
print("Strangers behind each baseline (embedding, words):")
print(strangers_used.to_string())
print()
summary = (pairs.assign(dimension=pd.Categorical(pairs["dimension"], DIMENSIONS, ordered=True),
                        pair=pd.Categorical(pairs["pair"], list(PAIR_NAMES.values()), ordered=True))
           .groupby(["study", "dimension", "pair"], observed=True)
           [["emb_distance", "emb_stranger", "word_distance", "word_stranger"]].mean().round(3))
print("Mean distances over every participant, by study, question and pair:")
print(summary.to_string())

Written: Cross-study/explanation_shift/nlp/explanation_shift_pairs.csv | 38121 rows | 12 columns

Strangers behind each baseline (embedding, words):
     emb_strangers_used  word_strangers_used
min                67.0                  0.0
50%               234.0                229.0
max               234.0                234.0

Mean distances over every participant, by study, question and pair:
                                            emb_distance  emb_stranger  word_distance  word_stranger
study   dimension          pair                                                                     
Study 2 transformativeness None-Slight             0.546         0.679          0.860          0.973
                           Slight-Dramatic         0.513         0.641          0.891          0.972
                           None-Dramatic           0.606         0.690          0.931          0.981
        essence_change     None-Slight             0.538         0.673          0.832          0.

In [8]:
def package_version(module_name):
    try:
        module = __import__(module_name)
    except Exception:
        return "not installed"
    return str(getattr(module, "__version__", "installed, version not recorded"))


environment = pd.DataFrame([
    ("run_id", RUN_ID),
    ("where", "Google Colab" if IN_COLAB else "local machine"),
    ("python", platform.python_version()),
    ("platform", platform.platform()),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scipy", package_version("scipy")),
    ("scikit-learn", package_version("sklearn")),
    ("sentence-transformers", package_version("sentence_transformers")),
    ("torch", package_version("torch")),
    ("embedding_model", "%s, %s" % (study_text.EMBEDDING_MODEL_NAME, EMBEDDING_STATUS)),
    ("embedding_device", DEVICE),
    ("embedding_numbers_per_text", str(size)),
    ("distinct_texts_embedded", str(len(texts))),
    ("explanations", str(len(blind))),
    ("complete_sets", str(len(complete_sets))),
    ("pairs", str(len(pairs))),
    ("seed_text_code", str(TEXT_SEED)),
    ("seeds_in_force", "; ".join("%s %s" % kv for kv in TEXT_SEEDS.items())),
], columns=["item", "value"])
environment.insert(0, "analysis", EXPLORATORY)
assert study_text.text_seeds() == TEXT_SEEDS
study_text.write_csv_atomic(ENVIRONMENT_PATH, environment, RUN_ID)
print("Status:", EXPLORATORY)
print(environment[["item", "value"]].to_string(index=False))
print()

completed_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
records = [("input", path) for path in INPUTS] + [("input", study_text.__file__)] + \
          [("output", PAIRS_PATH), ("output", ENVIRONMENT_PATH)]
for path in INPUTS:
    assert file_sha256(path) == INPUTS[path], "%s changed during the run." % show(path)
manifest = pd.DataFrame([dict(
    analysis=EXPLORATORY, run_id=RUN_ID, completed_utc=completed_utc, role=role,
    file=show(path), bytes=os.path.getsize(path), sha256=file_sha256(path),
) for role, path in records])
study_text.write_csv_atomic(MANIFEST_PATH, manifest, RUN_ID)
print(manifest[["role", "file", "bytes", "sha256"]].to_string(index=False))
print()
print("Manifest:", show(MANIFEST_PATH), "| written last.")

Status: exploratory: specified on 9 October 2026, after the registered results of all five studies were known; no registration names it
                      item                                                                                                               value
                    run_id                                                                                           20261014T191240Z-2e6d80f3
                     where                                                                                                       local machine
                    python                                                                                                              3.12.3
                  platform                                                                                        macOS-26.6.2-arm64-arm-64bit
                    pandas                                                                                                               2.2.2
      

---

## What this notebook does not do

It does not compare the arms. It never reads the label, so every number it prints is over all participants.
`ExplanationShift_02_Analysis.Rmd` joins the label on, defines the shift as the None to Dramatic distance
minus its stranger baseline, and asks whether the AI label changes it.

It keeps none of the 768 numbers per explanation. They exist only while the notebook runs, and only the
distances leave it.